# 군사기지 주변 지역의 주택 가격 예측하기

이 문제는 목표는 가상의 군사기지 주변 지역 정보를 활용해 그 지역의 주택 가격의 중간값을 예측하는 Regression 모델을 만드는 것입니다.  


## 데이터 셋 설명

주택 가격을 예측하는 학습을 하기 위한 훈련용 데이터와 나중에 만들어진 모델의 성능을 측정하기 위한 테스트 데이터가 별도의 파일로  제공이 됩니다.

 - 훈련용 데이터: train.csv
 - 테스트용 데이터: test.csv

훈련용 데이터와 테스트 데이터의 차이점은, 테스트 데이터의 경우 주택 가격의 중간값을 나타내는 median_house_value란 레이블 필드가 없고 일련번호 필드인 id가 존재한다는 점입니다.

### 훈련용 데이터

다음은 데이터셋(train.csv)에 들어있는 피쳐들과 그에 대한 설명들입니다. 이 정보는 주택별 정보가 아닌 지역별 정보입니다. 이를 바탕으로 지역별 주택 중간값을 예측합니다.

| 필드 | 타입 | 설명 |
| --- | --- | --- |
| housing_median_age | 숫자 | 해당 지역 주택 나이의 중간값  |
| total_rooms | 숫자 | 해당 지역 주택의 총 방의 수  |
| total_bedrooms | 숫자 | 해당 지역 주택의 총 침실의 수  |
| population | 숫자 | 해당 지역 인구 |
| households | 숫자 | 해당 지역 가구수 |
| median_income | 숫자 | 해당 지역 가구의 중간 소득 |
| ocean_proximity | 문자열 | 해당 지역으로부터 바닷가까지의 거리를 나타내는 문자열로 다음과 같은 값이 가능.<br> <left><li> "<1H OCEAN": 바다에서 1시간 이내의 거리에 있는 지역<br> <li> "INLAND": 내륙 지역으로 바다에서 멀리 떨어져 있는 지역<br><li>  "NEAR BAY": 만 근처에 있는 지역으로, 바다보다는 좀 더 작은 규모의 해안 지역이나 만과 가까운 위치<br> <li> "NEAR OCEAN": 바다와 가까운 지역으로, 해안 근처에 있는 주택</left>  |
| median_house_value | 숫자 | 예측 대상이 되는 주택 가격의 중간값 |

### 테스트용 데이터

아래는 테스트 데이터인 test.csv 파일의 예시입니다. train.csv 파일과 비교했을 때, 레이블 필드인 median\_house\_value가 없고 일련번호 필드인 id가 존재합니다.

| id |housing_median_age |total_rooms |total_bedrooms |population |households |median_income |ocean_proximity |
|  --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 9.0 | 5206.0 | 992.0 | 4660.0 | 978.0 | 2.885 | <1H OCEAN |
| 1 | 28.0 | 2608.0 | 576.0 | 1719.0 | 554.0 | 2.1186 | INLAND |
| 2 | 33.0 | 2731.0 | 456.0 | 1263.0 | 445.0 | 4.5568 | NEAR OCEAN |
| 3 | 36.0 | 1406.0 | 224.0 | 598.0 | 237.0 | 5.8964 | NEAR OCEAN |
| 4 | 18.0 | 1525.0 | 485.0 | 1705.0 | 448.0 | 3.7198 | <1H OCEAN |

### 초기코드 선택(Python)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 초기코드 - python
import pandas as pd

# 데이터 로드
train = pd.read_csv('./drive/MyDrive/maicon24/day1/Problem#2/data/train.csv')
test = pd.read_csv('./drive/MyDrive/maicon24/day1/Problem#2/data/test.csv')

train.head()

,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,median_house_value
0,43.0,1461.0,284.0,800.0,258.0,3.2279,<1H OCEAN,182400.0
1,33.0,2613.0,562.0,3150.0,543.0,4.3899,<1H OCEAN,180700.0
2,11.0,5363.0,1160.0,2783.0,1034.0,3.8583,<1H OCEAN,170700.0
3,43.0,2700.0,510.0,1146.0,506.0,5.1333,<1H OCEAN,345900.0
4,36.0,1407.0,385.0,1763.0,350.0,2.6364,<1H OCEAN,150000.0


### 제출 데이터

submission.csv 파일은 CSV 포맷으로 두 개의 필드로 구성됩니다 (id와 median_house_value). 아래 예가 있습니다.

|id|median_house_value|
|-|-|
|0|414800|
|1|294700|
|2|150000|
|3|78600|
|4|94400|

### 성능 평가

모델의 성능은 RMSE (Root Mean Square Error)으로 평가합니다. 70000 보다 작은 값이 나오면 괜찮다고 생각할 수 있습니다.

In [ ]:
import os
import numpy as np
import warnings; warnings.filterwarnings('ignore')
from sklearn.model_selection import KFold
from sklearn.ensemble import (HistGradientBoostingRegressor, RandomForestRegressor,
                              ExtraTreesRegressor, GradientBoostingRegressor)

# 데이터 경로: Colab(초기코드) 경로 우선, 없으면 로컬 ./data 사용
D = './drive/MyDrive/maicon24/day1/Problem#2/data/'
if not os.path.exists(D + 'train.csv'):
    D = './data/'
train = pd.read_csv(D + 'train.csv')
test = pd.read_csv(D + 'test.csv')

OCEAN = ['<1H OCEAN', 'INLAND', 'NEAR BAY', 'NEAR OCEAN']
def make_features(df):
    X = df.drop(columns=['id', 'median_house_value'], errors='ignore').copy()
    X['rooms_per_hh'] = X.total_rooms / X.households
    X['bed_ratio'] = X.total_bedrooms / X.total_rooms
    X['pop_per_hh'] = X.population / X.households
    X['bed_per_hh'] = X.total_bedrooms / X.households
    X['log_income'] = np.log1p(X.median_income)
    for c in OCEAN:                       # 고정 범주 목록 -> train/test 열 일치
        X['op_' + c] = (X.ocean_proximity == c).astype(int)
    return X.drop(columns='ocean_proximity')

X, y, X_test = make_features(train), train.median_house_value.values, make_features(test)
assert list(X.columns) == list(X_test.columns) and X.isna().sum().sum() == 0 and X_test.isna().sum().sum() == 0

# 앙상블 v6 (버전 비교에서 CV RMSE 가장 낮음): 6개 모델 단순 평균. (모델, 로그타깃 여부)
def build(seed):
    hgb = dict(learning_rate=0.05, max_iter=400, max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0)
    return {
        'hgb':       (HistGradientBoostingRegressor(random_state=seed, **hgb), False),
        'hgb_log':   (HistGradientBoostingRegressor(random_state=seed, **hgb), True),   # log(y) 로 학습 후 exp 복원
        'hgb2':      (HistGradientBoostingRegressor(learning_rate=0.03, max_iter=600, max_leaf_nodes=10,
                                                    min_samples_leaf=30, l2_regularization=3.0, random_state=seed), False),
        'rf':        (RandomForestRegressor(500, min_samples_leaf=2, max_features=0.5, n_jobs=-1, random_state=seed), False),
        'et':        (ExtraTreesRegressor(500, min_samples_leaf=2, max_features=0.7, n_jobs=-1, random_state=seed), False),
        'gbr_huber': (GradientBoostingRegressor(n_estimators=400, learning_rate=0.03, max_depth=4,
                                                subsample=0.8, loss='huber', random_state=seed), False),
    }

def fit_predict(model, logy, Xtr, ytr, Xte):
    pred = model.fit(Xtr, np.log(ytr) if logy else ytr).predict(Xte)
    return np.exp(pred) if logy else pred

rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))

# 5-Fold CV (OOF) 로 검증
oof = {k: np.zeros(len(X)) for k in build(0)}
for tr, va in KFold(5, shuffle=True, random_state=42).split(X):
    for k, (m, logy) in build(0).items():
        oof[k][va] = fit_predict(m, logy, X.iloc[tr], y[tr], X.iloc[va])
for k in oof:
    print(f'{k} CV RMSE: {rmse(oof[k], y):.1f}')
print(f'ensemble(v6) CV RMSE: {rmse(np.mean(list(oof.values()), axis=0), y):.1f}   (이전 v1 hgb+rf+et: 59103.5)')

# 전체 train으로 재학습 후 test 예측 (seed 평균)
preds = []
for seed in range(3):
    for k, (m, logy) in build(seed).items():
        preds.append(fit_predict(m, logy, X, y, X_test))
pred = np.mean(preds, axis=0)

### 결과 저장

채점을 위해, 위에서 구한 데이터를 현재 파일과 같은 디렉토리(.ipynb 파일이 있는 디렉토리)에 `submission.csv`이라는 이름으로 저장해주세요.

In [ ]:
# 제출 형식: id, median_house_value (index 열 없음)
df = pd.DataFrame({'id': test['id'], 'median_house_value': pred})
df.to_csv('submission.csv', index=False)

chk = pd.read_csv('submission.csv')
assert list(chk.columns) == ['id', 'median_house_value'] and len(chk) == len(test)
assert (chk['id'].values == test['id'].values).all() and np.isfinite(chk['median_house_value']).all()
chk.head()

In [ ]:
# (원본 예시 `df.to_csv('submission.csv')`는 index 열이 추가되어 제출 형식과 달라 사용하지 않음)
print(open('submission.csv').read().splitlines()[:3])

### 결과 테스트

해당 파트는 실제 예선전과 별개로 성능 확인을 위해 추가된 셀입니다.

In [ ]:
from sklearn.metrics import mean_squared_error

# 1. CSV 파일 불러오기
df1 = pd.read_csv('./drive/MyDrive/maicon24/day1/Problem#2/data/test_with_labels.csv')
df2 = pd.read_csv('submission.csv')

# 2. 마지막 컬럼 추출
col1 = df1.iloc[:, -1]
col2 = df2.iloc[:, -1]

# 3. RMSE 계산
rmse = np.sqrt(mean_squared_error(col1, col2))

# 4. 결과 출력
print("RMSE:", rmse)